In [151]:
import google.auth
import pymupdf
import re
import pandas as pd
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine
from google import genai
from google.genai import types
from collections import Counter
from langchain_postgres.vectorstores import PGVector
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from pgvector.sqlalchemy import Vector
from sqlalchemy import text

In [132]:
load_dotenv()

True

## Read Pdf

Analisar el patron del documento para maximizar los resultados de la busqueda y el resumen

In [121]:
def get_chapters(ruta_pdf: str, incluir_referencias: bool = True) -> list[dict]:
    """
    Fragmenta un artículo académico por capítulos con títulos en romanos y mayúsculas
    (I. INTRODUCCIÓN, II. CONCEPTUALIZACIÓN, ...). Pensado para PDFs a dos columnas.
    Retorna dicts: numero, etiqueta, titulo, contenido, pagina_inicio, pagina_fin.
    """
    romano_valido = re.compile(r"^M{0,3}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})$")
    patron_titulo = re.compile(r"^([IVXLC]+)\.\s+([^a-záéíóúñ]+)$")
    patron_referencias = re.compile(r"^(REFERENCIAS|BIBLIOGRAF[ÍI]A|REFERENCES)\s*$")
    valores = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100}

    def romano_a_entero(r):
        if not romano_valido.match(r):
            return None
        total = 0
        for i, c in enumerate(r):
            v = valores[c]
            total += -v if i + 1 < len(r) and v < valores[r[i + 1]] else v
        return total

    with pymupdf.open(ruta_pdf) as doc:
        paginas = [p.get_text() for p in doc]

    normalizar = lambda l: re.sub(r"\d+", "", l).strip()
    zona = lambda t: [normalizar(l) for l in t.splitlines()[:4] + t.splitlines()[-4:]]
    conteo = Counter(n for t in paginas for n in set(zona(t)) if n)
    repetidas = {l for l, c in conteo.items() if c >= 2 and len(l) >= 15}

    lineas = []
    for num, texto in enumerate(paginas, start=1):
        texto = re.sub(r"(\w)-\n(\w)", r"\1\2", texto)
        todas = texto.splitlines()
        for j, l in enumerate(todas):
            l = l.strip()
            en_zona = j < 4 or j >= len(todas) - 4
            if not l or l.isdigit() or (en_zona and normalizar(l) in repetidas):
                continue
            lineas.append((l, num))

    fragmentos, actual, siguiente, preambulo = [], None, 1, []
    i = 0
    while i < len(lineas):
        linea, pagina = lineas[i]
        m = patron_titulo.match(linea)
        es_titulo = m and romano_a_entero(m.group(1)) == siguiente
        es_refs = patron_referencias.match(linea) and actual is not None

        if es_titulo or es_refs:
            if actual:
                fragmentos.append(actual)
            if es_refs:
                if not incluir_referencias:
                    actual = None
                    break
                numero, etiqueta, titulo = None, "", linea.strip()
            else:
                numero, etiqueta, titulo = siguiente, m.group(1), m.group(2).strip()
                while i + 1 < len(lineas):
                    sig = lineas[i + 1][0]
                    if sig.isupper() and len(sig) < 60 and not patron_titulo.match(sig):
                        titulo += " " + sig
                        i += 1
                    else:
                        break
                siguiente += 1
            actual = {"numero": numero, "etiqueta": etiqueta, "titulo": titulo,
                      "contenido": "", "pagina_inicio": pagina, "pagina_fin": pagina}
        elif actual:
            actual["contenido"] += linea + "\n"
            actual["pagina_fin"] = pagina
        else:
            preambulo.append(linea)
        i += 1

    if actual:
        fragmentos.append(actual)

    for f in fragmentos:
        f["contenido"] = re.sub(r"(?<![.:])\n(?=[a-záéíóúñ(«])", " ", f["contenido"]).strip()

    if preambulo:
        fragmentos.insert(0, {"numero": 0, "etiqueta": "", "titulo": "Resumen / Preámbulo",
                              "contenido": "\n".join(preambulo),
                              "pagina_inicio": 1,
                              "pagina_fin": fragmentos[0]["pagina_inicio"] if fragmentos else 1})
    return fragmentos

In [122]:
path = r'/home/sebas/Documents/projects/opensource/rag-agent/agent/resources/Dialnet-UnaMiradaALaInteligenciaArtificial-7894426.pdf'

In [123]:
fragmentos = get_chapters(path)
df = pd.DataFrame(fragmentos)
columnas_en = {
    "numero": "number",
    "etiqueta": "label",
    "titulo": "title",
    "contenido": "content",
    "pagina_inicio": "start_page",
    "pagina_fin": "end_page",
}

capitulos = get_chapters(path)
df = pd.DataFrame(capitulos).rename(columns=columnas_en)
df

,number,label,title,content,start_page,end_page
0,0.0,,Resumen / Preámbulo,Vol. 2 / Núm. 3 / enero - junio de 2015; pág. ...,1,1
1,1.0,I,INTRODUCCIÓN,En este documento se describirá un estado del ...,1,1
2,2.0,II,CONCEPTUALIZACIÓN,"La palabra inteligencia, etimológicamente se d...",1,2
3,3.0,III,HISTORIA DE LA INTELIGENCIA ARTIFICIAL,"A través de los años, el ser humano se ha preo...",2,3
4,4.0,IV,CLASES DE INTELIGENCIA ARTIFICIAL,Existen diveras categorias de inteligencia hum...,3,4
5,5.0,V,APLICACIONES DE LA INTELIGENCIA ARTIFICIAL,La inteligencia artificial ha sido utilizada e...,4,5
6,6.0,VI,CONCLUSIONES,Con el anterior panorama general sobre intelig...,5,5
7,NaN,,REFERENCIAS,"[1]\nReal Academia Española, «Inteligencia,» D...",5,5


In [124]:
_,PROJECT_ID = google.auth.default()
PROJECT_ID

'superagentycs-dev'

In [157]:
genai_client = genai.Client(
    enterprise=True, 
    project=PROJECT_ID, 
    location='global'
)

In [ ]:
def get_summary(text : str) -> str:
    
    response = genai_client.models.generate_content(
        model="gemini-3.8-flash", 
        contents=f"Dame un resumen muy corto acerca de este capitulo : {text}"
    )

    return response.text

In [166]:
dimensionality = 768
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    project=PROJECT_ID,
    output_dimensionality=dimensionality,
    vertexai=True
)

## Summary Chapters

In [128]:
df['enhanced_summary'] = df['content'].apply(get_summary)
df.head()

,number,label,title,content,start_page,end_page,enhanced_summary
0,0.0,,Resumen / Preámbulo,Vol. 2 / Núm. 3 / enero - junio de 2015; pág. ...,1,1,Aquí tienes un resumen muy corto:\n\nEl artícu...
1,1.0,I,INTRODUCCIÓN,En este documento se describirá un estado del ...,1,1,Este documento presenta un estado del arte sob...
2,2.0,II,CONCEPTUALIZACIÓN,"La palabra inteligencia, etimológicamente se d...",1,2,**Resumen:**\n\nEl texto explica que la inteli...
3,3.0,III,HISTORIA DE LA INTELIGENCIA ARTIFICIAL,"A través de los años, el ser humano se ha preo...",2,3,**Resumen:**\n\nEl texto aborda la evolución h...
4,4.0,IV,CLASES DE INTELIGENCIA ARTIFICIAL,Existen diveras categorias de inteligencia hum...,3,4,"A diferencia de la **inteligencia humana**, qu..."


In [167]:
df["embedding"] = df["enhanced_summary"].apply(lambda t: embeddings.embed_query(t))
df.head()

,number,label,title,content,start_page,end_page,enhanced_summary,embedding
0,0.0,,Resumen / Preámbulo,Vol. 2 / Núm. 3 / enero - junio de 2015; pág. ...,1,1,Aquí tienes un resumen muy corto:\n\nEl artícu...,"[-0.026663352, 0.0015331028, -0.029846944, 0.0..."
1,1.0,I,INTRODUCCIÓN,En este documento se describirá un estado del ...,1,1,Este documento presenta un estado del arte sob...,"[-0.03691173, -0.017748224, -0.039401595, -0.0..."
2,2.0,II,CONCEPTUALIZACIÓN,"La palabra inteligencia, etimológicamente se d...",1,2,**Resumen:**\n\nEl texto explica que la inteli...,"[-0.014437694, 0.020469595, -0.018497491, 0.01..."
3,3.0,III,HISTORIA DE LA INTELIGENCIA ARTIFICIAL,"A través de los años, el ser humano se ha preo...",2,3,**Resumen:**\n\nEl texto aborda la evolución h...,"[-0.04380985, 0.0008681383, -0.02427634, -0.00..."
4,4.0,IV,CLASES DE INTELIGENCIA ARTIFICIAL,Existen diveras categorias de inteligencia hum...,3,4,"A diferencia de la **inteligencia humana**, qu...","[-0.002686019, 0.007830093, 0.0048449566, 0.02..."


In [161]:
PG_USER=os.environ.get("PG_USER")
PG_PASSWORD=str(os.environ.get("PG_PASSWORD"))
PG_HOST=str(os.environ.get("PG_HOST"))
PG_DATABASE=os.environ.get("PG_DATABASE")
PG_PORT=str(os.environ.get("PG_PORT"))

In [162]:
connection_string = f"postgresql://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
engine = create_engine(connection_string)

In [144]:
## RUN sudo apt install postgresql-1X-pgvector

In [163]:
with engine.begin() as conn:
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector"))

In [188]:
collection_name = "my_docs"
vector_store = PGVector(
    embeddings=embeddings,
    collection_name=collection_name,
    connection=engine,
    use_jsonb=True,
)

In [189]:
vector_store.add_embeddings(
    texts=df["enhanced_summary"].tolist(),
    embeddings=df["embedding"].apply(list).tolist(),
    metadatas=df[["label", "title", "content", "start_page", "end_page"]].to_dict("records"),
)

['8b174e9e-dca8-4838-83c9-1964655eb856',
 '9c41170b-9780-477c-b2eb-05df297cb2b6',
 '9c7a782a-4a2d-473c-aeab-81e345bf667c',
 '7f1c42b7-8cd4-4626-8738-19a8e5caf62f',
 '32d477c6-b429-4d54-a735-6c1ccab1ae61',
 '3bf8d2c3-44f8-4f5d-a89d-0b40bb899d16',
 '196fbd84-dd40-46dc-8b78-5a05fe8aea7a',
 '270012cc-3f78-4c62-9b1e-914bc1830732']

In [192]:
results = vector_store.similarity_search("Cual es la historia de la Inteligencia artificial", k=3)
response_documents = [i.metadata for i in results]
response_documents

[{'label': 'I',
  'title': 'INTRODUCCIÓN',
  'content': 'En este documento se describirá un estado del arte acerca de la inteligencia arficial como tendencia tecnológica que está impactando diferentes sectores.\nEn la sección II del documento se dará a conocer una definición de inteligencia artificial, su idea principal de estudio y el porqué surgió esta ciencia.\nEn la sección III se tiene la cronología de la Inteligencia Artificial, sus inicios, el desarrollo que ha tenido a traves de los años, y algunas de sus características.\n*\nMichael Estid Alvarado Rojas. Estudiante de Ingeniería de Sistemas. Corporación Universitaria Republicana.\nEn la sección IV se describen los dos tipos de inteligencia artificial que se tienen en cuenta en la actualidad, que son al Inteligencia Artificial convencional y la computacional, se habla a fondo de estos dos términos, sus parámetros y sus métodos de estudio.\nEn la sección V se resaltan aplicaciones de interés y se presentan algunas conclusiones s